# Compute and export waveform analysis

This section reads Kilosort and the raw recording, computes either good units or all units, and exports that analysis scope. The plotting section below only reads the exported files.

In [ ]:
from pathlib import Path
from typing import Any, Literal

import numpy as np
import spikeinterface as si
import spikeinterface.extractors as se
from numpy.typing import NDArray
from probeinterface import Probe
from spikeinterface.postprocessing.localization_tools import compute_center_of_mass

from Utils.si_utils import compute_template_ptp_summary, validate_data
from Utils.waveform_artifacts import (
    export_waveform_analysis,
    spike_position_analysis_dir,
    waveform_analysis_dir,
)

type FloatArray = NDArray[np.floating[Any]]
type IntArray = NDArray[np.integer[Any]]
type StructuredArray = NDArray[np.void]

In [ ]:
base_dir: Path = Path('/mnt/senzailab/Kai/#Recording/m14')
date: str = '260615'
session_id: str = '3'
probe_name: str = 'A'
session_dir: Path = base_dir / date / f'{date}_{session_id}'
output_dir: Path = session_dir / 'data'  # Parent of waveform/ and spike_position/; set another root to export elsewhere.
only_good_units: bool = True
unit_scope: Literal['good', 'all'] = 'good' if only_good_units else 'all'

pre_spike_ms: float = 0.5
post_spike_ms: float = 1.5
max_spikes_per_unit: int = 2_000
waveform_seed: int = 0
n_jobs: int = 32
overwrite_waveform_analyzer: bool = False

raw_num_channels: int = 384
raw_dtype: Literal['int16'] = 'int16'
raw_sampling_frequency: float = 30_000.0
gain_to_uv: float = 0.195
offset_to_uv: float = 0.0
probe_contact_radius_um: float = 6.0
unit_location_feature: Literal['ptp'] = 'ptp'
unit_location_radius_um: float = 75.0

kilosort_run: str = f'kilosort_{session_id}'
kilosort_dir: Path = session_dir / 'kilosort' / f'Probe{probe_name}' / kilosort_run
raw_file: Path = (
        next((session_dir / session_dir.parent.name).iterdir())
        / "experiment1"
        / "recording1"
        / "continuous"
        / f"OneBox-104.Probe{probe_name}"
        / "continuous.dat"
)
waveform_dir: Path = waveform_analysis_dir(output_dir, probe_name)
spike_position_dir: Path = spike_position_analysis_dir(output_dir, probe_name)
analysis_dir: Path = waveform_dir  # Plotting reads waveform + its linked spike-position file.
waveform_analyzer_folder: Path = (
    session_dir / 'data' / 'spikeinterface_analyzer' / f'Probe{probe_name}'
)

validated_data = validate_data(
    recording_file=raw_file,
    kilosort_dir=kilosort_dir,
)

sorting: si.BaseSorting = se.read_kilosort(kilosort_dir, keep_good_only=only_good_units)
unit_ids: IntArray = sorting.unit_ids
channel_map: IntArray = validated_data.channel_map
channel_positions: FloatArray = validated_data.channel_positions
channel_shank_ids: IntArray = validated_data.channel_shank_ids

recording: si.BaseRecording = si.read_binary(
    file_paths=raw_file,
    sampling_frequency=raw_sampling_frequency,
    dtype=raw_dtype,
    num_channels=raw_num_channels,
    gain_to_uV=gain_to_uv,
    offset_to_uV=offset_to_uv,
    is_filtered=False,
)
recording = recording.select_channels(channel_map)
probe = Probe(ndim=2, si_units='um')
probe.set_contacts(
    channel_positions,
    shapes='circle',
    shape_params={'radius': probe_contact_radius_um},
)
probe.set_device_channel_indices(np.arange(raw_num_channels))
recording = recording.set_probe(probe)
recording_duration_minutes: float = recording.get_total_duration() / 60.0

print(f'{unit_scope.title()} units: {len(unit_ids)}')
print(f'Recording: {recording_duration_minutes:.2f} min · channels: {recording.get_num_channels()}')

## Build or load the waveform analyzer

The analyzer uses the same SpikeInterface binary-folder cache as before, now stored inside the session data directory.

In [ ]:
selection_params = {
    'method': 'uniform',
    'max_spikes_per_unit': max_spikes_per_unit,
    'margin_size': None,
    'seed': waveform_seed,
}

waveform_analyzer: si.SortingAnalyzer
if waveform_analyzer_folder.exists() and not overwrite_waveform_analyzer:
    waveform_analyzer = si.load_sorting_analyzer(waveform_analyzer_folder)
    if not np.array_equal(waveform_analyzer.unit_ids, sorting.unit_ids):
        raise ValueError('Cached analyzer unit IDs do not match the current sorting.')
    if not np.array_equal(waveform_analyzer.channel_ids, recording.channel_ids):
        raise ValueError('Cached analyzer channel IDs do not match the current recording.')
    if waveform_analyzer.sparsity is not None:
        raise ValueError('Cached analyzer is sparse; rebuild it with overwrite_waveform_analyzer = True.')
    print(f'Loaded waveform analyzer: {waveform_analyzer_folder}')
else:
    waveform_analyzer_folder.parent.mkdir(parents=True, exist_ok=True)
    waveform_analyzer = si.create_sorting_analyzer(
        sorting,
        recording,
        format='binary_folder',
        folder=waveform_analyzer_folder,
        sparse=False,
        overwrite=overwrite_waveform_analyzer,
        n_jobs=n_jobs,
        chunk_duration='1s',
        progress_bar=True,
    )

selection_extension = waveform_analyzer.get_extension('random_spikes')
if not isinstance(selection_extension, si.ComputeRandomSpikes) or selection_extension.params != selection_params:
    waveform_analyzer.compute('random_spikes', **selection_params)
    selection_extension = waveform_analyzer.get_extension('random_spikes')
if not isinstance(selection_extension, si.ComputeRandomSpikes):
    raise TypeError('Unexpected random_spikes extension type.')
if waveform_analyzer.has_extension('waveforms'):
    raise ValueError('This analyzer must not store individual waveforms.')

template_extension = waveform_analyzer.get_extension('templates')
template_params_match = (
        isinstance(template_extension, si.ComputeTemplates)
        and template_extension.params.get('operators') == ['average']
        and template_extension.params.get('ms_before') == pre_spike_ms
        and template_extension.params.get('ms_after') == post_spike_ms
)
if not template_params_match:
    template_extension = waveform_analyzer.compute(
        'templates',
        ms_before=pre_spike_ms,
        ms_after=post_spike_ms,
        operators=['average'],
        n_jobs=n_jobs,
        chunk_duration='1s',
        progress_bar=True,
    )
    template_extension = waveform_analyzer.get_extension('templates')
if not isinstance(template_extension, si.ComputeTemplates):
    raise TypeError('Unexpected templates extension type.')
if waveform_analyzer.has_extension('unit_locations'):
    waveform_analyzer.delete_extension('unit_locations')

selected_spikes: StructuredArray = selection_extension.get_random_spikes()
print(f'Selected {len(selected_spikes):,} waveform spikes across {len(unit_ids)} units.')

## Calculate templates, PTP, channels, and waveform time

Every numerical result reused by the plotting section is calculated once here and exported below.

In [ ]:
template_array: FloatArray = template_extension.get_templates(operator='average')
time_ms: FloatArray = (
                              np.arange(template_array.shape[1]) - template_extension.nbefore
                      ) / waveform_analyzer.sampling_frequency * 1_000.0
template_ptp_summary = compute_template_ptp_summary(template_array)
channel_locations: FloatArray = waveform_analyzer.get_channel_locations()

print(f'Templates: {template_array.shape} · dtype: {template_array.dtype}')

## Calculate unit locations

Center of mass uses each unit's raw-template PTP within the configured radius around its maximum-PTP channel.

In [ ]:
location_sparsity_mask = np.zeros(
    (len(unit_ids), waveform_analyzer.get_num_channels()),
    dtype=bool,
)
for unit_index in range(len(unit_ids)):
    best_channel_index = int(template_ptp_summary.best_channel_indices[unit_index])
    distances = np.linalg.norm(
        channel_locations - channel_locations[best_channel_index],
        axis=1,
    )
    location_sparsity_mask[unit_index] = distances <= unit_location_radius_um

location_sparsity = si.ChannelSparsity(
    mask=location_sparsity_mask,
    unit_ids=unit_ids,
    channel_ids=waveform_analyzer.channel_ids,
)
dense_location_templates = si.Templates(
    templates_array=template_array,
    sampling_frequency=waveform_analyzer.sampling_frequency,
    nbefore=template_extension.nbefore,
    is_in_uV=True,
    channel_ids=waveform_analyzer.channel_ids,
    unit_ids=unit_ids,
    probe=waveform_analyzer.get_probe(),
)
unit_locations: FloatArray = compute_center_of_mass(
    dense_location_templates.to_sparse(location_sparsity),
    feature=unit_location_feature,
)

## Export analysis artifacts

`waveform/` and `spike_position/` are sibling output roots. Each keeps its own `config.json` and `run.log`; probe data lives below `Probe*`.

In [ ]:
exported_paths = export_waveform_analysis(
    output_dir,
    source_session=session_dir.name,
    source_probe=probe_name,
    source_kilosort_dir=kilosort_dir,
    source_raw_file=raw_file,
    unit_scope=unit_scope,
    sorting=sorting,
    unit_ids=unit_ids,
    selected_spikes=selected_spikes,
    template_array=template_array,
    template_ptp_summary=template_ptp_summary,
    unit_locations=unit_locations,
    channel_ids=np.asarray(waveform_analyzer.channel_ids),
    channel_locations=channel_locations,
    channel_shank_ids=channel_shank_ids,
    sampling_frequency=raw_sampling_frequency,
    recording_num_frames=recording.get_num_frames(),
    recording_duration_minutes=recording_duration_minutes,
    time_ms=time_ms,
    nbefore=template_extension.nbefore,
    pre_spike_ms=pre_spike_ms,
    post_spike_ms=post_spike_ms,
    max_spikes_per_unit=max_spikes_per_unit,
    waveform_seed=waveform_seed,
    unit_location_feature=unit_location_feature,
    unit_location_radius_um=unit_location_radius_um,
    run_config={
        'output_dir': str(output_dir),
        'waveform_analyzer_folder': str(waveform_analyzer_folder),
        'n_jobs': n_jobs,
        'overwrite_waveform_analyzer': overwrite_waveform_analyzer,
        'raw_dtype': raw_dtype,
        'raw_num_channels': raw_num_channels,
        'gain_to_uv': gain_to_uv,
        'offset_to_uv': offset_to_uv,
        'probe_contact_radius_um': probe_contact_radius_um,
    },
)
print(f'Exported waveform data: {exported_paths.waveform_dir}')
print(f'Exported spike positions: {exported_paths.spike_position_dir}')

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

from typing import Literal

from Utils.probe_plotting import WaveformUnitPlotCollection

if 'analysis_dir' not in globals():
    raise RuntimeError('Run the compute configuration cell first so analysis_dir uses its output_dir.')

plot_unit_ids: list[int] = [520, 115]

heatmap_baseline_end_ms: float = -0.25
probe_ptp_scale: Literal['per_unit', 'global_uv'] = 'per_unit'
probe_contact_radius_um: float = 6.0
save_figures: bool = False

unit_plots = WaveformUnitPlotCollection(
    analysis_dir,
    plot_unit_ids,
    heatmap_baseline_end_ms=heatmap_baseline_end_ms,
    probe_ptp_scale=probe_ptp_scale,
    probe_contact_radius_um=probe_contact_radius_um,
    save_figures=save_figures,
    output_dir=analysis_dir / 'figures',
)
unit_plots.print_load_summary()
unit_plots.load_success

In [ ]:
unit_plots.list.plot_local_average_heatmaps(
    local_channel_mode='same_x_column',
    local_channel_count=5,
)
# unit_plots.list.plot_best_channel_averages()
# unit_plots.list.plot_ptp_gradients()
# unit_plots.plot_unit_locations(show_other_units=False)
# max_contact_results = unit_plots.list.plot_max_ptp_contacts(
#     layout='panels',
#     show_other_units=False,
# )

In [ ]:
spike_selection_results = unit_plots.list.plot_waveform_spike_selection_times()
for result in spike_selection_results:
    print(result.message)

## Local average waveforms

## PTP gradients

## Unit locations

## Maximum-PTP contacts

Each configured unit is drawn in a separate probe figure.

In [ ]:

for result in max_contact_results:
    print(result.message)